In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Some global variables

In [2]:
filter_start = "2025-10-01"
filter_end = "2026-10-01"

period_start = "2023-10-01"
period_end = "2026-10-01"

In [3]:
# Set up exchange rate dataframe for every day in the period

exchange_rate = yf.Ticker("CAD=X").history(start = period_start, end = period_end)["Close"]

# Keep only year month date, datetime objects, in index column of dataframe
exchange_rate.index = pd.to_datetime(exchange_rate.index.strftime('%Y-%m-%d'))

display(exchange_rate.head())

Date
2023-10-02    1.35737
2023-10-03    1.36763
2023-10-04    1.37091
2023-10-05    1.37442
2023-10-06    1.37057
Name: Close, dtype: float64

Extracting stocks

In [4]:
# EXTRACTING TICKERS. MAKE SURE TO CHANGE FILE NAME FOR ACTUAL SUBMISSION
tickers = pd.read_csv("Tickers.csv", header=None)[0].to_list()
close_prices = pd.DataFrame()

for tick in tickers:
    t = yf.Ticker(tick)
    information = t.info

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock
    if (information["market"] != "ca_market") and (information["market"] != "us_market"):
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information["quoteType"] != "EQUITY":
        print(tick, "not stock")
        continue

    if (information["currency"] != "CAD") and (information["currency"] != "USD"):
        print(tick, "currency wrong")
        continue
    

    temp = yf.download(tick, filter_start, filter_end)

    if temp["Volume"].mean()[tick] < 5000:
        print(tick, "not enough volume")
        continue
    
    close_prices[tick] = temp["Close"]

# Filters out the first and last trading day of each month
idx = close_prices.index.to_series()
date_series_first=idx.groupby([idx.dt.year, idx.dt.month]).first()
date_series_last=idx.groupby([idx.dt.year, idx.dt.month]).last()
monthly_first = []
monthly_last = []
for i in date_series_first:
    monthly_first.append(i)
for i in date_series_last:
    monthly_last.append(i)

for i in range(len(monthly_first)):
    start_ind = close_prices.index.get_loc(monthly_first[i])
    end_ind = close_prices.index.get_loc(monthly_last[i])

    if (end_ind - start_ind + 1) < 18:
        close_prices = close_prices.drop(close_prices.loc[monthly_first[i]:monthly_last[i]].index)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$AGN: possibly delisted; no timezone found


AGN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$BK: possibly delisted; no price data found  (1d 1927-12-21 -> 2026-11-26)


BK doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$CELG: possibly delisted; no timezone found


CELG doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$MON: possibly delisted; no timezone found


MON doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$RTN: possibly delisted; no timezone found


RTN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [5]:
display(close_prices.head())

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,254.316605,235.580231,129.656540,236.889038,77.725349,220.630005,324.080444,215.199997,49.622257,6.67,...,65.924812,163.093597,208.619995,20.261112,107.826408,174.982285,339.266174,229.894775,78.023796,46.106377
2025-10-02,255.989151,228.041824,129.190262,237.501389,77.725349,222.410004,325.974243,217.429993,49.426430,6.63,...,68.169777,165.405457,211.240005,19.976391,106.874405,176.854431,344.545654,229.679321,79.124161,46.135265
2025-10-03,256.875183,225.487259,130.744537,238.453964,79.072166,219.509995,325.737518,216.300003,49.583088,6.28,...,68.486244,165.728714,224.889999,20.104973,108.913048,174.914383,350.857544,231.912247,80.243004,46.289337
2025-10-06,255.551117,221.901199,129.918823,241.224197,81.024086,220.899994,327.463654,219.729996,49.338306,6.68,...,70.503754,165.180145,229.520004,19.893730,108.951515,176.359711,349.464600,232.509674,80.095070,45.952305
2025-10-07,255.342041,224.446136,129.219406,244.198547,82.029327,221.779999,323.498444,221.820007,49.240391,6.48,...,73.787132,162.084610,225.100006,19.875360,108.605324,171.742416,354.227814,227.074234,79.503273,45.740452


In [6]:
full_data = pd.DataFrame()

cols = {}
for tick in close_prices:
    prices = yf.download(tick, period_start, period_end)["Close"][tick]
    if yf.Ticker(tick).info["currency"] == "USD":
        prices = prices * exchange_rate.reindex(prices.index).ffill()
    cols[tick] = prices
    
full_data = pd.concat(cols, axis=1, sort=True).sort_index()

full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

full_data = full_data.ffill()

display(full_data.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,232.519145,180.319200,121.325747,393.013010,76.241147,175.725132,195.350702,254.954813,33.728007,6.02,...,141.522591,73.839996,17.100691,70.040115,198.282192,655.916213,257.465500,174.719966,38.394415,45.243141
2023-10-03,232.456366,180.677246,122.319391,393.376016,75.013737,170.570816,190.994155,258.372658,32.977460,5.98,...,141.424380,70.440002,17.204716,69.378128,196.408622,654.324061,260.870999,174.530883,37.721732,45.229576
2023-10-04,234.716907,181.430038,122.433567,398.388435,76.116891,174.105576,193.537665,255.990027,33.094829,5.86,...,141.326139,72.230003,17.756868,69.195206,197.781094,657.373793,259.558983,174.915328,37.691544,45.243141
2023-10-05,237.011665,181.598940,123.452851,399.757000,77.327376,173.121948,195.441646,256.040701,33.333052,5.20,...,141.623504,71.510002,18.100964,69.997543,194.697560,666.297724,258.126512,174.975263,38.550104,45.247677
2023-10-06,239.834034,182.060504,123.977207,401.894164,78.033794,175.378129,197.848890,256.817403,33.252435,5.15,...,142.615900,73.269997,18.044947,70.359024,196.517888,675.474363,259.738355,175.497596,38.430051,45.256702


Sharpe calculation

In [7]:
half_life = 250

In [8]:
def get_stats(tick, half_life):
    s = full_data[tick].dropna()
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return average, standard_deviation

In [9]:
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

sharpe = {}

for tick in close_prices:
    stats = get_stats(tick, half_life)
    sharpe[tick] = (stats[0] - Rf)/stats[1]

sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

Risk free return: 0.00010671471670599812
{'T.TO': np.float64(-0.09193760481902127), 'ACN': np.float64(-0.03509386977226787), 'PEP': np.float64(-0.03230760718287938), 'UPS': np.float64(-0.019860269360688527), 'ABT': np.float64(-0.017787466611563888), 'PYPL': np.float64(-0.014007788173990567), 'BA': np.float64(-0.01316657200407587), 'UNH': np.float64(-0.008824855240501419), 'CL': np.float64(0.0031530149151281404), 'AIG': np.float64(0.003682114079208524), 'LMT': np.float64(0.00811416738167635), 'AXP': np.float64(0.016000904862675358), 'QCOM': np.float64(0.018020355958977094), 'BLK': np.float64(0.022920164123996507), 'AMZN': np.float64(0.026520473622096554), 'PFE': np.float64(0.02705317200945819), 'BIIB': np.float64(0.028587261021987623), 'SHOP.TO': np.float64(0.029318168385391155), 'UNP': np.float64(0.03063043826464172), 'BMY': np.float64(0.03198046715482833), 'LLY': np.float64(0.03970524412895659), 'BAC': np.float64(0.04076946644505066), 'TXN': np.float64(0.04130309356692816), 'USB': np.

In [10]:
def weighted_corr(t1, t2):
    d1 = full_data[t1].pct_change(fill_method=None)
    d2 = full_data[t2].pct_change(fill_method=None)
    return d1.ewm(halflife=half_life).corr(d2).dropna().iloc[-1]

Correlation matrix

In [11]:
corr_mat = pd.DataFrame()

for t1 in close_prices:
    for t2 in close_prices:
        corr_mat.loc[t1, t2] = weighted_corr(t1, t2)

In [12]:
display(corr_mat)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
AAPL,1.000000,0.194254,0.132854,0.215325,0.277563,0.279193,0.355878,0.312301,0.340458,0.113994,...,0.312324,0.317230,0.338428,0.194442,0.211207,0.277675,-0.001601,0.246692,0.300749,0.359230
ABBV,0.194254,1.000000,0.337372,0.186326,0.253049,-0.053363,0.157285,0.060093,0.162880,-0.014198,...,0.144141,0.010036,0.028752,0.074433,0.053394,0.071150,0.139082,0.254595,0.226953,0.205309
ABT,0.132854,0.337372,1.000000,0.267510,0.246026,-0.007405,0.146429,0.115855,0.176013,-0.145556,...,0.168768,-0.041628,0.116199,0.089898,0.034980,-0.012994,0.106915,0.291407,0.181667,0.212030
ACN,0.215325,0.186326,0.267510,1.000000,0.225876,0.162441,0.322218,0.139568,0.185739,0.062243,...,0.344292,0.069002,0.365620,0.222115,0.024306,-0.085630,0.110766,0.165151,0.174303,0.272432
AIG,0.277563,0.253049,0.246026,0.225876,1.000000,0.114573,0.318293,0.213833,0.354936,-0.012067,...,0.174231,0.093021,0.149236,0.025890,0.170660,0.132204,0.123403,0.352018,0.191967,0.359135
AMZN,0.279193,-0.053363,-0.007405,0.162441,0.114573,1.000000,0.401539,0.331861,0.316599,0.191711,...,0.302479,0.281235,0.344012,-0.197242,0.209296,0.243408,0.050052,0.149052,0.197262,0.313900
AXP,0.355878,0.157285,0.146429,0.322218,0.318293,0.401539,1.000000,0.411301,0.667997,0.193264,...,0.387069,0.308753,0.412547,0.036176,0.394655,0.289368,0.178149,0.333913,0.333174,0.700108
BA,0.312301,0.060093,0.115855,0.139568,0.213833,0.331861,0.411301,1.000000,0.360954,0.118260,...,0.269480,0.317297,0.232243,-0.000760,0.251276,0.321524,0.110067,0.187327,0.271724,0.346611
BAC,0.340458,0.162880,0.176013,0.185739,0.354936,0.316599,0.667997,0.360954,1.000000,0.160196,...,0.271996,0.278173,0.282604,-0.022011,0.444876,0.291087,0.122073,0.349899,0.334237,0.769056
BB.TO,0.113994,-0.014198,-0.145556,0.062243,-0.012067,0.191711,0.193264,0.118260,0.160196,1.000000,...,0.146720,0.255018,0.224028,-0.046085,0.208502,0.221540,0.051965,0.092423,0.179718,0.186197


In [13]:
period = 15
threshold = 1.5

In [14]:
# Note: this shouldn't be used as a raw value, as there may be periods where the variance
# jumps are spread across two periods and thus not counted. Use a threshold lower than
# what you're truly looking for. 
def count_jumps_std(period, threshold):
    result = pd.DataFrame()
    std_dict = {}
    for tick in close_prices:
        result.loc[tick, "Occurrences"] = 0
        std_dict[tick] = full_data[tick].std()
    interval = [len(full_data)-period, len(full_data)]
    while interval[0] >= 0:
        for tick in close_prices:
            if full_data[tick].iloc[interval[0]:interval[1]].std() >= std_dict[tick] * threshold:
                result.loc[tick, "Occurrences"]+=1
        interval[0]-=period
        interval[1]-=period

    return result

count_jumps_std(period, threshold)

,Occurrences
AAPL,0.0
ABBV,0.0
ABT,0.0
ACN,0.0
AIG,0.0
AMZN,0.0
AXP,0.0
BA,0.0
BAC,0.0
BB.TO,0.0


In [15]:
period = 15
threshold = 0.8

In [16]:
# Note: this shouldn't be used as a raw value, as there may be periods where the price
# jumps are spread across two periods and thus not counted. Use a threshold lower than
# what you're truly looking for. 

def count_price_plummets(period, threshold):
    result = pd.DataFrame()
    std_dict = {}
    for tick in close_prices:
        result.loc[tick, "Occurrences"] = 0
    interval = [len(full_data)-period, len(full_data)-1]
    while interval[0] >= 0:
        for tick in close_prices:
            if full_data.iloc[interval[1]][tick] <= full_data.iloc[interval[0]][tick] * threshold:
                result.loc[tick, "Occurrences"]+=1
        interval[0]-=period
        interval[1]-=period

    return result

count_price_plummets(period, threshold)

,Occurrences
AAPL,0.0
ABBV,0.0
ABT,0.0
ACN,0.0
AIG,0.0
AMZN,0.0
AXP,0.0
BA,0.0
BAC,0.0
BB.TO,3.0


In [17]:
sectors = {}

for tick in close_prices:
    sectors[tick] = yf.Ticker(tick).info.get("sector")

High growth

In [18]:
def get_best_in_sector():
    valid_ticks = sorted([x for x in close_prices], key=lambda x: (-sharpe[x]))

    sector_chosen = {}
    
    for tick in valid_ticks:
        sector = sectors[tick]
        
        if (sector in sector_chosen) and len(sector_chosen[sector]) == 3:
            continue
        else:
            if sector not in sector_chosen:
                sector_chosen[sector] = [tick]
            else:
                sector_chosen[sector].append(tick)

    return sector_chosen

get_best_in_sector()

{'Financial Services': ['TD.TO', 'RY.TO', 'C'],
 'Industrials': ['CAT', 'UNP', 'LMT'],
 'Consumer Defensive': ['KO', 'PM', 'MO'],
 'Technology': ['AAPL', 'BB.TO', 'TXN'],
 'Healthcare': ['MRK', 'ABBV', 'LLY'],
 'Consumer Cyclical': ['AMZN'],
 'Communication Services': ['T.TO']}

In [19]:
#Momentum
def get_momentum():
    prices = full_data[close_prices.columns]
    
    return prices.asof(pd.Timestamp("2026-09-01")) / prices.asof(pd.Timestamp("2025-09-01")) - 1


In [20]:
#Dividends
def get_dividends():
    result = pd.Series()
    for tick in close_prices:
        div = yf.Ticker(tick).dividends
        div.index = div.index.tz_localize(None).normalize()

        div = div[(div.index >= period_start) & (div.index < period_end)]

        if div.empty:
            result[tick] = 0
            continue

        if yf.Ticker(tick).info.get("currency") == "USD":
            div = (div * exchange_rate.asof(div.index))

        price = full_data[tick].asof(div.index)

        yields = div/price

        age = (pd.Timestamp(filter_end) - div.index).days
        weights = 0.5**(age/half_life)

        result[tick] = np.sum(yields*weights)

    return result

In [21]:
#Positive return?
def sharpe_check():
    result = pd.Series()
    for tick in close_prices:
        result[tick] = min(0, sharpe[tick])

    return result

sharpe_check()

AAPL       0.000000
ABBV       0.000000
ABT       -0.017787
ACN       -0.035094
AIG        0.000000
AMZN       0.000000
AXP        0.000000
BA        -0.013167
BAC        0.000000
BB.TO      0.000000
BIIB       0.000000
BLK        0.000000
BMY        0.000000
C          0.000000
CAT        0.000000
CL         0.000000
KO         0.000000
LLY        0.000000
LMT        0.000000
MO         0.000000
MRK        0.000000
PEP       -0.032308
PFE        0.000000
RY.TO      0.000000
PM         0.000000
PYPL      -0.014008
QCOM       0.000000
SHOP.TO    0.000000
T.TO      -0.091938
TD.TO      0.000000
TXN        0.000000
UNH       -0.008825
UNP        0.000000
UPS       -0.019860
USB        0.000000
dtype: float64

In [22]:
#Revenue geometric mean
def revenue_CAGR():
    result = pd.Series()
    for tick in close_prices:
        stmt = yf.Ticker(tick).income_stmt
        if "Total Revenue" not in stmt.index:
            result[tick] = np.nan
            continue

        
        rev = stmt.loc["Total Revenue"].dropna().sort_index()

        if len(rev) < 2:
            result[tick] = np.nan
            continue

        years = (rev.index[-1] - rev.index[0]).days/365.25
        result[tick] = pow((rev.iloc[-1]/rev.iloc[0]), (1/years)) - 1

    return result

revenue_CAGR()

AAPL       0.076995
ABBV       0.017521
ABT        0.005127
ACN        0.049828
AIG       -0.036936
AMZN       0.117285
AXP        0.109637
BA         0.103306
BAC        0.060017
BB.TO      0.014234
BIIB      -0.009351
BLK        0.106518
BMY        0.014488
C          0.045886
CAT        0.043822
CL         0.042925
KO         0.036882
LLY        0.316791
LMT        0.043829
MO        -0.008923
MRK        0.031215
PEP        0.028253
PFE       -0.147945
RY.TO      0.109007
PM         0.085681
PYPL       0.064254
QCOM       0.063763
SHOP.TO    0.273061
T.TO       0.036102
TD.TO      0.102209
TXN       -0.040668
UNH        0.113494
UNP       -0.004914
UPS       -0.040394
USB        0.056744
dtype: float64

In [23]:
#Revenue geometric mean
def revenue_std():
    result = pd.Series()
    for tick in close_prices:
        stmt = yf.Ticker(tick).income_stmt
        if "Total Revenue" not in stmt.index:
            result[tick] = np.nan
            continue
        
        rev_delta = stmt.loc["Total Revenue"].dropna().sort_index().pct_change().dropna()

        result[tick] = rev_delta.std()
        if len(rev_delta) <= 2:
            result[tick] = np.nan

    return result

revenue_std()

AAPL       0.066323
ABBV       0.076550
ABT        0.076677
ACN        0.033166
AIG        0.027233
AMZN       0.006985
AXP        0.030292
BA         0.248082
BAC        0.027089
BB.TO      0.369836
BIIB       0.028381
BLK        0.098018
BMY        0.051376
C          0.011817
CAT        0.081048
CL         0.035597
KO         0.023793
LLY        0.125724
LMT        0.017401
MO         0.006045
MRK        0.031085
PEP        0.027785
PFE        0.256037
RY.TO      0.049920
PM         0.018812
PYPL       0.019574
QCOM       0.083581
SHOP.TO    0.024350
T.TO       0.049199
TD.TO      0.020437
TXN        0.142734
UNH        0.034844
UNP        0.022367
UPS        0.048701
USB        0.092225
dtype: float64

In [24]:
def z_score(ser):
    z = (ser-ser.mean())/ser.std()
    return z.fillna(z.max())

z_score(revenue_std())

AAPL      -0.015182
ABBV       0.114919
ABT        0.116526
ACN       -0.436980
AIG       -0.512460
AMZN      -0.770036
AXP       -0.473541
BA         2.297005
BAC       -0.514292
BB.TO      3.845854
BIIB      -0.497852
BLK        0.388018
BMY       -0.205327
C         -0.708562
CAT        0.172129
CL        -0.406056
KO        -0.556216
LLY        0.740463
LMT       -0.637533
MO        -0.781988
MRK       -0.463456
PEP       -0.505429
PFE        2.398196
RY.TO     -0.223850
PM        -0.619578
PYPL      -0.609886
QCOM       0.204362
SHOP.TO   -0.549133
T.TO      -0.233020
TD.TO     -0.598913
TXN        0.956854
UNH       -0.415633
UNP       -0.574360
UPS       -0.239359
USB        0.314315
dtype: float64